# Diabetes Progression: Regression Analysis

**Author:** Bilal Ahmad — Healthcare Data Analyst

> **Note:** This notebook is for learning and portfolio demonstration only. It is **not** clinical or medical advice, and it uses **no real patient data**. Dataset: the built-in scikit-learn diabetes dataset (a standard teaching dataset of ten baseline variables and a quantitative disease-progression measure).


## 1. Introduction

Instead of classifying a disease, this notebook *predicts a number*: how far diabetes progresses one year after baseline measurements. Two regression models are compared — ordinary Linear Regression and Ridge regression — and judged with MAE, RMSE and R-squared.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="progression")
print("Dataset shape:", X.shape)
print("Target (progression) summary:")
print(y.describe().round(2))
print("\nMissing values:", int(X.isna().sum().sum()))

Dataset shape: (442, 10)
Target (progression) summary:
count    442.00
mean     152.13
std       77.09
min       25.00
25%       87.00
50%      140.50
75%      211.50
max      346.00
Name: progression, dtype: float64

Missing values: 0


## 2. Exploratory Data Analysis

First, which baseline variables are most strongly associated with progression? Correlation is a starting point for discussion, not evidence of causation.

In [2]:
X.describe().T[["mean", "std", "min", "max"]].round(3)
corr = pd.concat([X, y], axis=1).corr()["progression"].drop("progression")
print("Correlation with progression, strongest first:")
print(corr.sort_values(key=abs, ascending=False).round(3))

Correlation with progression, strongest first:
bmi    0.586
s5     0.566
bp     0.441
s4     0.430
s3    -0.395
s6     0.382
s1     0.212
age    0.188
s2     0.174
sex    0.043
Name: progression, dtype: float64


## 3. Train / Test Split and Modelling

Ridge regression adds a penalty on large coefficients, which helps when predictors are correlated with each other — common in health data.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)
models = {"Linear Regression": LinearRegression(), "Ridge (alpha=1.0)": Ridge(alpha=1.0)}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"--- {name} ---")
    print("MAE :", round(mean_absolute_error(y_test, pred), 2))
    print("RMSE:", round(float(np.sqrt(mean_squared_error(y_test, pred))), 2))
    print("R^2 :", round(r2_score(y_test, pred), 4))

--- Linear Regression ---
MAE : 42.79
RMSE: 53.85
R^2 : 0.4526
--- Ridge (alpha=1.0) ---
MAE : 46.14
RMSE: 55.47
R^2 : 0.4192


## 4. Residual Discussion

Residuals (actual minus predicted) should scatter randomly around zero. A visible pattern — for example, residuals growing with the predicted value — would warn that the linear model is missing something. The plot and the summary below let us check.

In [4]:
best = Ridge(alpha=1.0).fit(X_train, y_train)
pred = best.predict(X_test)
resid = y_test - pred
print("Residual mean:", round(resid.mean(), 2))
print("Residual std :", round(resid.std(), 2))
plt.figure(figsize=(7, 4))
plt.scatter(pred, resid, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted progression")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual plot — Ridge regression")
plt.tight_layout()
plt.show()

Residual mean: -0.54
Residual std : 55.79


## 5. Conclusion

**Results (from running the code above):** Linear Regression gave MAE 42.79, RMSE 53.85 and R-squared 0.4526; Ridge (alpha = 1.0) gave MAE 46.14, RMSE 55.47 and R-squared 0.4192, so the simpler linear model performed slightly better on this split. The strongest correlations with progression were bmi (0.586) and s5 (0.566). Residuals for the Ridge model had mean -0.54 and standard deviation 55.79, i.e. predictions were typically off by roughly 43-56 progression units.

R-squared on a teaching dataset like this is typically modest, which is itself the lesson: baseline measurements explain only part of disease progression, and the rest reflects factors the data do not capture (treatment, lifestyle, genetics, measurement noise). A responsible report states the error (MAE/RMSE) alongside R-squared, checks the residuals, and never presents a regression prediction as a medical prognosis.